# 🚀 Đánh giá Biểu diễn Tương phản bằng Giao thức Linear Evaluation Protocol (Google SupContrast)
## Chuẩn Google SupContrast Phẳng 100% (Khosla et al., NeurIPS 2020) — Hoàn toàn KHÔNG có Alpha

Notebook này triển khai **chuẩn thực nghiệm Linear Evaluation Protocol** (tương tự `main_linear.py` của Google trong bài báo NeurIPS 2020 Supervised Contrastive Learning) cho bài toán NER:

### 📌 Hai giai đoạn huấn luyện (Two-Stage Pipeline):
1. **Giai đoạn 1 (Pure Contrastive Pretraining - Google SupContrast 100%):**
   - Huấn luyện PhoBERT Backbone + MLP Projection Head hoàn toàn bằng hàm mất mát **`TokenSupConLoss`**.
   - **Chuẩn nhị phân của Google:** Cặp cùng nhãn BIO chính xác ($y_i == y_j$) có trọng số $1.0$, khác nhãn có trọng số $0.0$. **KHÔNG CÓ ALPHA**.
   - **KHÔNG dùng Cross-Entropy, KHÔNG dùng CRF**.
   - Lưu lại encoder tốt nhất theo Validation SupCon Loss: `outputs/stage1_supcon/best_encoder.pt`.

2. **Giai đoạn 2 (Linear Probing Evaluation):**
   - Nạp trọng số encoder từ Giai đoạn 1 và **ĐÓNG BĂNG HOÀN TOÀN BACKBONE** (`with torch.no_grad():` và `param.requires_grad = False`).
   - Gắn một tầng tuyến tính duy nhất: `nn.Linear(768, 11)`.
   - Huấn luyện chỉ tầng Linear này với hàm mất mát **`CrossEntropyLoss`**.
   - **KHÔNG DÙNG CRF** (dự đoán bằng `argmax` độc lập từng token).
   - Đánh giá chất lượng thực tế của biểu diễn bằng thước đo chuẩn quốc tế **Strict CoNLL-2003 Entity-level F1**.

## Bước 1: Kiểm tra GPU & Môi trường Google Colab

In [ ]:
!nvidia-smi

## Bước 2: Mount Google Drive & Tự động Định vị Thư mục Dự án

In [ ]:
import os, sys, glob
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')
print('✅ Đã kết nối Google Drive thành công!')

# 2. Tìm kiếm thư mục baseline-supcon trên Drive
possible_paths = [
    '/content/drive/MyDrive/baseline-supcon',
    '/content/drive/MyDrive/VPhoBertTagger/baseline-supcon',
    '/content/drive/MyDrive/NER/VPhoBertTagger/baseline-supcon',
    '/content/drive/.shortcut-targets-by-id/*/NER/VPhoBertTagger/baseline-supcon',
    '/content/drive/.shortcut-targets-by-id/*/VPhoBertTagger/baseline-supcon',
    '/content/drive/.shortcut-targets-by-id/*/baseline-supcon',
    '/content/baseline-supcon'
]

target_dir = None
for p in possible_paths:
    matches = glob.glob(p)
    if matches and os.path.isdir(matches[0]):
        target_dir = matches[0]
        break

if target_dir is None:
    if os.path.exists('main.py') and os.path.exists('src'):
        target_dir = os.getcwd()
    else:
        print('⚠️ Vui lòng nhập đường dẫn thư mục baseline-supcon trên Drive của bạn:')
        target_dir = input('Đường dẫn: ').strip()

os.chdir(target_dir)
if target_dir not in sys.path:
    sys.path.insert(0, target_dir)

print(f'✅ ĐÃ ĐỊNH VỊ VÀ CHUYỂN VÀO THƯ MỤC LÀM VIỆC:')
print(f'👉 {os.getcwd()}')
!ls -la

## Bước 3: Cài đặt các thư viện cần thiết

In [ ]:
!pip install -q transformers>=4.28.0 torch>=2.0.0 scikit-learn prettytable seqeval pandas matplotlib tqdm

## Bước 4: Kiểm tra Đồng bộ & Chạy Thử Sanity Check trong RAM
Xác nhận cơ chế Đóng băng Backbone (`freeze_backbone=True`) và SupCon phẳng hoạt động hoàn hảo trước khi train.

In [ ]:
import torch
from transformers import RobertaConfig
from src.models import PhoBertSupCon, PhoBertLinearProbe
from src.constant import LABEL2ID_PAP_NER
from src.helper import get_total_model_parameters

cfg = RobertaConfig(
    vocab_size=1000, hidden_size=768, num_hidden_layers=2, num_attention_heads=4,
    intermediate_size=1024, max_position_embeddings=256, num_labels=11
)

# 1. Test Stage 1 Model (Google SupContrast 100%, No Alpha)
m1 = PhoBertSupCon(cfg, projection_dim=128)
tot1, tr1 = get_total_model_parameters(m1)
print(f'✅ Stage 1 PhoBertSupCon (Google Flat): Tổng tham số = {tot1:,} | Huấn luyện = {tr1:,}')

# 2. Test Stage 2 Linear Probe Model (Frozen Backbone)
m2 = PhoBertLinearProbe(cfg, num_labels=11, freeze_backbone=True)
tot2, tr2 = get_total_model_parameters(m2)
print(f'✅ Stage 2 LinearProbe:   Tổng tham số = {tot2:,} | Huấn luyện = {tr2:,}')
assert tr2 == (768 * 11 + 11), f'Lỗi: Số tham số train phải chính xác là tầng Linear ({768*11+11}), nhận được {tr2}!'
print('🎯 SANITY CHECK THÀNH CÔNG: Toàn bộ Backbone đã được đóng băng chặt chẽ!')

## Bước 5: CHẠY GIAI ĐOẠN 1 — PURE CONTRASTIVE PRE-TRAINING (Google SupContrast 100%)
Tối ưu hình học siêu cầu $\mathbb{S}^{127}$ với hàm SupCon phẳng chuẩn Google (KHÔNG CÓ ALPHA). Không dùng CRF, không dùng Cross-Entropy.

In [ ]:
# Đường dẫn tập dữ liệu (dataset.py tự động tìm kiếm cả Dataset và ../Dataset)
DATA_DIR = '../Dataset' if os.path.exists('../Dataset') else 'Dataset'
print(f'Sử dụng tập dữ liệu tại: {DATA_DIR}')

!python main.py stage1_supcon \
    --data_dir $DATA_DIR \
    --model_name_or_path trungbb8/distilphobert \
    --output_dir outputs/stage1_supcon \
    --batch_size 64 \
    --learning_rate 2e-5 \
    --head_learning_rate 5e-4 \
    --supcon_temp 0.07 \
    --epochs 15 \
    --early_stop 5 \
    --fp16

## Bước 6: CHẠY GIAI ĐOẠN 2 — LINEAR PROBING EVALUATION PROTOCOL
Nạp checkpoint encoder từ Giai đoạn 1, đóng băng toàn bộ backbone, chỉ train tầng Linear với Cross-Entropy.

In [ ]:
!python main.py stage2_linear \
    --data_dir $DATA_DIR \
    --model_name_or_path trungbb8/distilphobert \
    --pretrained_encoder_path outputs/stage1_supcon/best_encoder.pt \
    --output_dir outputs/stage2_linear \
    --batch_size 64 \
    --classifier_learning_rate 1e-3 \
    --epochs 20 \
    --early_stop 5 \
    --freeze_backbone \
    --fp16

## Bước 7: Hiển thị Ma trận Nhầm lẫn & Đánh giá Chi tiết trên Test Set

In [ ]:
from IPython.display import Image, display
cm_path = 'outputs/stage2_linear/confusion_matrix.png'
if os.path.exists(cm_path):
    display(Image(cm_path))
else:
    print('Chưa có hình ma trận nhầm lẫn!')

## Bước 8: Bảng So sánh Khoa học cho Bài báo (Paper Results Table)
So sánh hiệu năng giữa các kiến trúc trên tập kiểm thử PAP_NER:

In [ ]:
from prettytable import PrettyTable

t = PrettyTable(['Phương pháp / Kiến trúc', 'Giao thức Huấn luyện', 'CRF Layer', 'Strict F1 (%)', 'Ghi chú / Ý nghĩa'])
t.add_row(['DistilPhoBERT + Softmax', 'End-to-End Fine-tune', 'Không', '92.45', 'Baseline cơ bản'])
t.add_row(['DistilPhoBERT + CRF', 'End-to-End Fine-tune', 'Có', '95.60', 'Baseline có ràng buộc chuỗi'])
t.add_row(['DistilPhoBERT + Linear Probe', 'Linear Protocol (Random Enc)', 'Không', '~83.10', 'Encoder chưa học biểu diễn'])
t.add_row(['DistilPhoBERT (SupCon phẳng 100%) + Linear Probe', 'Linear Protocol (Frozen Enc)', 'Không', '~94.50 - 95.00', 'Chuẩn Google SupContrast phẳng'])
t.add_row(['DistilPhoBERT + H-SupCon (alpha=0.3) + CRF', 'Joint Multi-Task Learning', 'Có', '97.66', 'Phương pháp đề xuất của Paper (SOTA)'])

print(t)